## Final Workflow: ML based integration pipeline (Sorted Neighbourhood Blocker)

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

### Schema Mapping

In [2]:
from PyDI.io import load_parquet, load_csv

amazon_books = load_parquet(
    DATA_DIR / "amazon.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks.parquet",
  name="metabooks"
)

In [3]:
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util


# embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def similarity_score(src, tgt):
    """Compute semantic + fuzzy similarity between column names."""
    emb_src = model.encode(src, convert_to_tensor=True)
    emb_tgt = model.encode(tgt, convert_to_tensor=True)
    semantic_sim = util.cos_sim(emb_src, emb_tgt).item()
    fuzzy_sim = fuzz.token_sort_ratio(src, tgt) / 100.0
    return 0.6 * semantic_sim + 0.4 * fuzzy_sim


def schema_match_semantic(source_cols, target_cols, threshold=0.6):
    """Find best target column for each source column."""
    mapping = {}
    for src in source_cols:
        best_tgt, best_score = None, 0
        for tgt in target_cols:
            score = similarity_score(src, tgt)
            if score > best_score:
                best_score, best_tgt = score, tgt
        mapping[src] = best_tgt if best_score >= threshold else None
    return mapping

/Users/abd/Developer/team_project_data_integration/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def unify_schemas(schema_dict, target_schema_name="metabooks"):
    """
    Use one dataset (e.g., 'metabooks') as target schema.
    Match all other schemas to it.
    """
    if target_schema_name not in schema_dict:
        raise ValueError(f"Target schema '{target_schema_name}' not found in schema_dict")

    target_cols = schema_dict[target_schema_name]
    mappings = {}

    for name, cols in schema_dict.items():
        if name == target_schema_name:
            # Target schema maps to itself
            mappings[name] = {col: col for col in cols}
        else:
            mappings[name] = schema_match_semantic(cols, target_cols)

    return target_cols, mappings

In [5]:
schemas = {
    "amazon": amazon_books.columns,
    "metabooks": metabooks.columns,
    "goodreads": goodreads.columns
}

target_schema, mappings = unify_schemas(schemas, target_schema_name="goodreads")

print("Target Schema:")
print(target_schema)

print("\nSchema Mappings:")
for name, mapping in mappings.items():
    print(f"\n{name}:")
    for k, v in mapping.items():
        print(f"  {k:20} -> {v}")

Target Schema:
Index(['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres',
       'bookformat', 'edition', 'page_count', 'publisher', 'publish_year',
       'price', 'isbn_clean'],
      dtype='object')

Schema Mappings:

amazon:
  id                   -> id
  title                -> title
  book-author          -> author
  year-of-publication  -> publish_year
  publisher            -> publisher
  isbn_clean           -> isbn_clean

metabooks:
  id                   -> id
  title                -> title
  author_name          -> author
  average_rating       -> rating
  num_rating           -> numratings
  language             -> language
  genres               -> genres
  publisher            -> publisher
  page_count           -> page_count
  price                -> price
  publish_year         -> publish_year
  isbn_clean           -> isbn_clean

goodreads:
  id                   -> id
  title                -> title
  author               -> author
  rating       

In [6]:
def apply_schema_mapping(df, mapping):
    rename_dict = {k: v for k, v in mapping.items() if v is not None}
    return df.rename(columns=rename_dict)

amazon_books = apply_schema_mapping(amazon_books, mappings["amazon"])
metabooks = apply_schema_mapping(metabooks, mappings["metabooks"])
goodreads = apply_schema_mapping(goodreads, mappings["goodreads"])

### Entity Matching

In [7]:
train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [8]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators_m2a = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    
    StringComparator(column='isbn_clean',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='isbn_clean',similarity_function='levenshtein', preprocess=str.lower),

    NumericComparator(column='publish_year',max_difference=1)
]

comparators_m2g = comparators_m2a + [
    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower,
                     list_strategy='best_match'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="rating", max_difference=0.2)
]

### Sorted Neighbourhood Blocker

In [9]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker


blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)


blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

### ML-based Matcher

In [10]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)

train_features_m2a = feature_extractor_m2a.create_features(
    metabooks, amazon_books, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks, goodreads, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [12]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [13]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a = ml_matcher_m2a.match(
    metabooks, amazon_books,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g = ml_matcher_m2g.match(
    metabooks, goodreads,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [14]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)

### Data Fusion

In [15]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

amazon_books.attrs["trust_score"] = 3
metabooks.attrs["trust_score"] = 2
goodreads.attrs["trust_score"] = 1

correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a.csv")
correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g.csv")

all_correspondences = pd.concat([correspondences_m2a, correspondences_m2g], ignore_index=True)

len(all_correspondences)

31814

In [16]:
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', longest_string)
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('isbn_clean', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('rating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres', union)

In [17]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_books, metabooks, goodreads],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')
display(fused_ml_snblocker.head())

Fused rows: 30,185


,_id,_fusion_group_id,_fusion_sources,numratings,title,id,rating,price,publish_year,language,genres,isbn_clean,page_count,author,publisher,_fusion_confidence,_fusion_metadata,edition,bookformat
0,metabooks_401477,group_0,"[metabooks, amazon_books]",107,eyes wide shut a screenplay,metabooks_401477,4.4,62.18,1999.0,English,"[['Humor', 'Entertainment', 'Movies']]",0446676322,281.0,arthur schnitzler,Warner Books,0.842741,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN
1,amazon_123817,group_1,"[metabooks, amazon_books]",6,a secret history of time to come,amazon_123817,4.5,4.50,1979.0,English,None,0394501667,303.0,robie macauley,Knopf,0.727273,"{'title_rule': 'longest_string', 'title_source...",NaN,NaN
2,metabooks_347208,group_2,"[metabooks, amazon_books]",492,rest in pieces mrs murphy mysteries paperback,metabooks_347208,4.6,7.99,1993.0,English,"[['Mystery, Thriller', 'Suspense', 'Mystery']]",0553562398,384.0,sneaky pie brown,Bantam,0.833965,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN
3,amazon_107993,group_3,"[metabooks, amazon_books]",160,liberated parents liberated children your guid...,amazon_107993,4.8,6.99,1990.0,English,"[['Self-Help', 'Relationships']]",0380711346,272.0,adele faber,Perennial Currents,0.818182,"{'title_rule': 'longest_string', 'title_source...",NaN,NaN
4,metabooks_355604,group_4,"[metabooks, amazon_books]",32,the complete idiots guide to getting rich 2nd ...,metabooks_355604,4.7,NaN,1999.0,English,"[['Business', 'Money', 'Personal Finance']]",0028629523,368.0,larry waschka,Alpha Communications,0.734991,"{'numratings_rule': 'first_non_null', 'numrati...",NaN,NaN


In [18]:
fused_ml_snblocker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30185 entries, 0 to 30184
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 30185 non-null  object 
 1   _fusion_group_id    30185 non-null  object 
 2   _fusion_sources     30185 non-null  object 
 3   numratings          30185 non-null  int64  
 4   title               30185 non-null  object 
 5   id                  30185 non-null  object 
 6   rating              30185 non-null  float64
 7   price               27975 non-null  float64
 8   publish_year        30171 non-null  float64
 9   language            30069 non-null  object 
 10  genres              28964 non-null  object 
 11  isbn_clean          30185 non-null  object 
 12  page_count          27473 non-null  float64
 13  author              30185 non-null  object 
 14  publisher           30185 non-null  object 
 15  _fusion_confidence  30185 non-null  float64
 16  _fus

In [19]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")